# Task 1 · Notebook 01 — Data Collection

**Source:** [Skytrax – British Airways reviews](https://www.airlinequality.com/airline-reviews/british-airways/)

**Dataset:** `data/raw/skytrax_ba_reviews_2026-02.csv`, 4,018 reviews scraped on 27 February 2026 with `requests` and `BeautifulSoup` (flights from July 2014 to February 2026). The scraping code is in `scraper_feb2026.py` in this folder.

Each review has a title, traveller details (aircraft, traveller type, seat type, route, date flown) and six 1–5 star ratings.

**Note on re-scraping (November 2026):** An attempt to re-scrape with the full review text failed. Skytrax's review pages return HTTP 404 from South African connections while the home page loads normally, which points to regional restrictions. The February 2026 scrape is therefore used as-is and kept untouched in `data/raw/`. All cleaning happens in Notebook 02.

**Limitation:** Only review *titles* were captured, not the full review body, so text analysis in this project is based on titles.

In [1]:
from pathlib import Path
import pandas as pd

In [2]:
RAW_FILE = Path("data/raw/skytrax_ba_reviews_2026-02.csv")

df = pd.read_csv(RAW_FILE)
print(f"Rows: {len(df):,}   Columns: {df.shape[1]}")
df.head()

Rows: 4,018   Columns: 12


,Review Title,Aircraft,Type of Traveller,Seat Type,Route,Date Flown,Seat Comfort,Cabin Staff Service,Food Beverages,Inflight Entertainment,Ground Service,Value For Money
0,"""fairly horrendous experience""",No Aircraft,Couple Leisure,Business Class,Orlando to Gatwick,February 2026,3,1,1,3,1,2
1,"""get a better service from Jet2 or EasyJet""",A320,Family Leisure,Economy Class,Sharm El Sheikh to London Gatwick,February 2026,2,1,No rating,No rating,1,2
2,"""don't honour what their staff say""",A380,Couple Leisure,Economy Class,Johannesburg to London,February 2026,1,5,3,1,3,1
3,“experience was deeply disappointing”,No Aircraft,Family Leisure,Economy Class,Brussels to Dubai via London,February 2026,No rating,No rating,No rating,No rating,1,1
4,"""Outstanding cabin crew""",Boeing 777-200,Solo Leisure,Business Class,Mumbai to London Heathrow,February 2026,5,5,4,3,2,5


In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 4018 entries, 0 to 4017
Data columns (total 12 columns):
 #   Column                  Non-Null Count  Dtype
---  ------                  --------------  -----
 0   Review Title            4018 non-null   str  
 1   Aircraft                4018 non-null   str  
 2   Type of Traveller       4018 non-null   str  
 3   Seat Type               4018 non-null   str  
 4   Route                   4018 non-null   str  
 5   Date Flown              4018 non-null   str  
 6   Seat Comfort            4018 non-null   str  
 7   Cabin Staff Service     4018 non-null   str  
 8   Food Beverages          4018 non-null   str  
 9   Inflight Entertainment  4018 non-null   str  
 10  Ground Service          4018 non-null   str  
 11  Value For Money         4018 non-null   int64
dtypes: int64(1), str(11)
memory usage: 797.5 KB


In [4]:
# The old scraper wrote placeholders such as "No Aircraft" / "No rating" for empty fields.
# Count them per column so we know how much is really missing.
placeholder_counts = (
    df.astype(str)
      .apply(lambda col: col.str.startswith("No ").sum())
)

missing_report = pd.DataFrame({
    "placeholders": placeholder_counts,
    "% of rows": (placeholder_counts / len(df) * 100).round(1),
})
missing_report.sort_values("placeholders", ascending=False)

,placeholders,% of rows
Aircraft,1917,47.7
Inflight Entertainment,1298,32.3
Ground Service,856,21.3
Date Flown,778,19.4
Route,776,19.3
Type of Traveller,771,19.2
Food Beverages,468,11.6
Cabin Staff Service,151,3.8
Seat Comfort,134,3.3
Seat Type,2,0.0


In [5]:
# Flag rows where each detail field is a placeholder
no_route    = df["Route"].astype(str).str.startswith("No ")
no_date     = df["Date Flown"].astype(str).str.startswith("No ")
no_traveller = df["Type of Traveller"].astype(str).str.startswith("No ")

all_three = no_route & no_date & no_traveller
print("Missing Route:", no_route.sum())
print("Missing all three (Route + Date Flown + Traveller):", all_three.sum())

# Where do these rows sit in the file? (the file is ordered newest -> oldest)
print("Position of first such row:", all_three.idxmax(), "of", len(df))
print("Share of these rows in the last 800 rows of the file:", round(all_three.iloc[-800:].mean() * 100, 1), "%")

Missing Route: 776
Missing all three (Route + Date Flown + Traveller): 768
Position of first such row: 3250 of 4018
Share of these rows in the last 800 rows of the file: 96.0 %


## First look at data quality

The scraper stored empty fields as placeholder text (e.g. `"No Aircraft"`, `"No rating"`). Counting these per column:

| Column | Missing | % |
|---|---|---|
| Aircraft | 1,917 | 47.7% |
| Inflight Entertainment | 1,298 | 32.3% |
| Ground Service | 856 | 21.3% |
| Date Flown / Route / Type of Traveller | ~775 each | ~19% |
| Food & Beverages | 468 | 11.6% |
| Cabin Staff Service | 151 | 3.8% |
| Seat Comfort | 134 | 3.3% |
| Seat Type | 2 | 0.0% |
| Value For Money, Review Title | 0 | 0% |

**Finding:** Route, Date Flown and Type of Traveller are missing *together* in 768 rows, and 96% of the last 800 rows of the file (the oldest reviews) are affected. These are older reviews published before Skytrax recorded those fields. They are not random gaps.

**Implications for cleaning (Notebook 02):**
- Placeholders will be converted to proper missing values (`NaN`).
- Missing values will **not** be filled with made-up numbers. Analyses that need route, date or traveller type will simply use the reviews that have them.
- `Aircraft` is missing for almost half of the reviews, so aircraft-level conclusions will be treated as indicative only.
- `Value For Money` is complete, which makes it a reliable signal for later analysis.

In [6]:
titles = df["Review Title"].str.strip('"').str.strip()

# Skytrax used a generic title for older reviews instead of a real headline
generic = titles.str.contains("british airways customer review", case=False)
print("Generic 'British Airways customer review' titles:", generic.sum())
print("...of which in the oldest 800 rows:", generic.iloc[-800:].sum())
print("Reviews with a real, informative title:", (~generic).sum())

# Exact duplicate rows (every column identical)
print("\nExact duplicate rows:", df.duplicated().sum())
print("Duplicates that have a generic title:", df[df.duplicated()].pipe(lambda d: generic[d.index]).sum())

print("\nMost common titles:")
titles.value_counts().head(8)

Generic 'British Airways customer review' titles: 958
...of which in the oldest 800 rows: 800
Reviews with a real, informative title: 3060

Exact duplicate rows: 244
Duplicates that have a generic title: 241

Most common titles:


Review Title
British Airways customer review    958
never fly British Airways again      6
I was pleasantly surprised           5
Overall a good flight                5
Terrible customer service            3
missed my connecting flight          3
one of the worst airlines            3
a very comfortable flight            3
Name: count, dtype: int64

## Review titles

| | Count |
|---|---|
| Generic title "British Airways customer review" | 958 |
| ...of which among the oldest 800 reviews | 800 (all of them) |
| Reviews with a real, informative title | **3,060** |

The oldest reviews were given a generic site title, not a passenger-written headline, so they contain **no usable text**.

**Exact duplicate rows:** 244, but 241 of them have the generic title. Because those older reviews lack route, date and traveller type, two different passengers with identical star ratings look the same. Only 3 rows are likely true duplicates.

## Summary & decisions for Notebook 02

1. Convert all placeholder strings to `NaN`; do not invent values.
2. **Text analysis / sentiment model:** use only the reviews with informative titles (3,060, or 3,057 once Notebook 02 removes the 3 true duplicates).
3. **Rating analysis:** keep all reviews; generic-title rows still carry genuine star ratings.
4. Drop only true duplicates (identical rows that also have an informative title).
5. Treat `Aircraft` (48% missing) as indicative only.

The raw file in `data/raw/` stays untouched; Notebook 02 saves a cleaned copy to `data/processed/`.